In [4]:
import pandas as pd

ml_table = pd.read_parquet(
    "data/artifacts/01_ml_table.parquet"
)

print("Shape:", ml_table.shape)

print("\nOrder status:")
print(ml_table["order_status"].value_counts(dropna=False))

Shape: (99441, 41)

Order status:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [5]:
display(ml_table.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_lat,customer_lng,...,avg_product_width_cm,total_product_volume_cm3,avg_product_photos_qty,avg_product_name_length,avg_product_description_length,seller_customer_distance_km,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,-23.576170,-46.587276,...,13.0,1976.0,4.0,40.0,268.0,18.681711,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,-12.126651,-45.008162,...,19.0,4693.0,1.0,29.0,178.0,861.035367,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,-16.744472,-48.514624,...,21.0,9576.0,1.0,46.0,232.0,514.547140,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,-5.774611,-35.273916,...,20.0,6000.0,3.0,59.0,468.0,1821.802656,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,-23.675316,-46.515116,...,15.0,11475.0,4.0,38.0,316.0,29.593095,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


In [7]:

delivered = ml_table[
    ml_table["order_status"] == "delivered"
].copy()

print("عدد delivered:", len(delivered))

print(
    "Delivered بدون تاريخ تسليم:",
    delivered["order_delivered_customer_date"].isna().sum()
)

print(
    "Delivered مع تاريخ تسليم:",
    delivered["order_delivered_customer_date"].notna().sum()
)

عدد delivered: 96478
Delivered بدون تاريخ تسليم: 8
Delivered مع تاريخ تسليم: 96470


In [ ]:
import pandas as pd


# 1. Convert date/time columns


date_features = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for feature in date_features:
    ml_table[feature] = pd.to_datetime(
        ml_table[feature],
        errors="coerce"
    )



# 2. Keep valid delivered orders


labeled_orders = ml_table[
    (ml_table["order_status"] == "delivered") &
    (ml_table["order_delivered_customer_date"].notna()) &
    (ml_table["order_estimated_delivery_date"].notna())
].copy()



# 3. Create target label


labeled_orders["is_delayed"] = (
    labeled_orders["order_delivered_customer_date"]
    > labeled_orders["order_estimated_delivery_date"]
).astype(int)



# 4. Check results


print("Shape:", labeled_orders.shape)

print("\nTarget distribution:")
print(labeled_orders["is_delayed"].value_counts())

print("\nDate columns data types:")
print(labeled_orders[date_features].dtypes)

print("عدد الطلبات المستخدمة:", len(labeled_orders))

labeled_orders["is_delayed"] = (
    labeled_orders["order_delivered_customer_date"]
    > labeled_orders["order_estimated_delivery_date"]
).astype(int)

print("\nOrder status:")
print(labeled_orders["order_status"].value_counts())

print("\nMissing delivered date:")
print(labeled_orders["order_delivered_customer_date"].isna().sum())

print("\nMissing estimated date:")
print(labeled_orders["order_estimated_delivery_date"].isna().sum())

print("\nTarget:")
print(labeled_orders["is_delayed"].value_counts())

Shape: (96470, 42)

Target distribution:
is_delayed
0    88644
1     7826
Name: count, dtype: int64

Date columns data types:
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object
عدد الطلبات المستخدمة: 96470

Order status:
order_status
delivered    96470
Name: count, dtype: int64

Missing delivered date:
0

Missing estimated date:
0

Target:
is_delayed
0    88644
1     7826
Name: count, dtype: int64


In [8]:
labeled_orders.to_parquet(
    "data/artifacts/02_labeled_orders1.parquet",
    index=False
)

print("\nSaved successfully.")


Saved successfully.


In [9]:
check = pd.read_parquet(
    "data/artifacts/02_labeled_orders1.parquet"
)

print(check.shape)
print(check["order_status"].value_counts())
print(check["is_delayed"].value_counts())

(96470, 42)
order_status
delivered    96470
Name: count, dtype: int64
is_delayed
0    88644
1     7826
Name: count, dtype: int64
